In [1]:
# Install dependencies
!pip -q install -U \
    transformers sentence-transformers langchain langchain-community \
    langchain-huggingface langchain-openai faiss-cpu datasets ragas \
    accelerate fastapi uvicorn bitsandbytes huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 612.9/612.9 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/

In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ─── Paths & Configuration ────────────────────────────────────────────────────

BASE_DIR         = "/content/drive/MyDrive/Research_My_Component/cardiology"
MODELS_DIR       = f"{BASE_DIR}/models"
VECTORSTORE_DIR  = f"{BASE_DIR}/vectorstore/faiss_index"

HYDE_PATH        = f"{MODELS_DIR}/hyde-sciFive-cardiology-generator"
INSTRUCTOR_PATH  = f"{MODELS_DIR}/instructor-large"
INBEDDER_PATH    = f"{MODELS_DIR}/inbedder-roberta-large"
PROJECTOR_PATH   = f"{MODELS_DIR}/projector.pt"

DATA_PATH            = "/content/drive/MyDrive/Research_My_Component/miriad_cardiology.json"
PIPELINE_OUTPUT_PATH = f"{BASE_DIR}/eval_results_pipeline_first100.csv"
FINAL_OUTPUT_PATH    = f"{BASE_DIR}/eval_final_scored_first100.csv"

EVAL_SAMPLE_SIZE = 100
TOP_K            = 5
ALPHA            = 0.5
DOMAIN_MAX_DISTANCE_TEXT = 0.25   # slightly relaxed from 0.22 → better recall

# IMPROVED: Llama-3.2-3B-Instruct replaces FLAN-T5-Base
FINAL_LLM_MODEL          = "meta-llama/Llama-3.2-3B-Instruct"
FINAL_LLM_MAX_NEW_TOKENS = 300
FINAL_LLM_NUM_BEAMS      = 1   # greedy works better for instruct models

print("BASE_DIR:", BASE_DIR)
print("DATA_PATH:", DATA_PATH)
print("VECTORSTORE_DIR:", VECTORSTORE_DIR)
print("FINAL_LLM_MODEL:", FINAL_LLM_MODEL)

BASE_DIR: /content/drive/MyDrive/Research_My_Component/cardiology
DATA_PATH: /content/drive/MyDrive/Research_My_Component/miriad_cardiology.json
VECTORSTORE_DIR: /content/drive/MyDrive/Research_My_Component/cardiology/vectorstore/faiss_index
FINAL_LLM_MODEL: meta-llama/Llama-3.2-3B-Instruct


In [ ]:
# ─── HuggingFace Login (required for gated Llama model) ──────────────────────

from huggingface_hub import login

HF_TOKEN = ""
login(token=HF_TOKEN)
print("✓ Logged in to HuggingFace Hub")

✓ Logged in to HuggingFace Hub


In [5]:
# ─── Imports ──────────────────────────────────────────────────────────────────

import os, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from functools import lru_cache
from typing import List, Tuple, Optional
from tqdm.notebook import tqdm

from transformers import (
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from sentence_transformers import SentenceTransformer

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.document import Document as LC_Document

from datasets import Dataset
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import AnswerCorrectness, Faithfulness, AnswerRelevancy
from langchain_openai import ChatOpenAI
from openai import OpenAI

print("✓ All imports successful")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✓ All imports successful


/tmp/ipykernel_715/1600894995.py:28: DeprecationWarning: Importing AnswerCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerCorrectness
  from ragas.metrics import AnswerCorrectness, Faithfulness, AnswerRelevancy
/tmp/ipykernel_715/1600894995.py:28: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import AnswerCorrectness, Faithfulness, AnswerRelevancy
/tmp/ipykernel_715/1600894995.py:28: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import AnswerCorrectness, Faithfulness, AnswerRelevan

In [6]:
# ─── API Keys ─────────────────────────────────────────────────────────────────

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("✓ Loaded OPENAI_API_KEY from Colab Secrets")
except Exception as e:
    print("⚠️  Could not load OPENAI_API_KEY from Colab Secrets.")
    print("    Create a Colab secret named OPENAI_API_KEY, then re-run.")
    print("    This is only required for the RAGAS / judge evaluation stage.")
    print("    Details:", e)

openai_client = OpenAI()

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

✓ Loaded OPENAI_API_KEY from Colab Secrets
Device: cuda


In [7]:
# ─── Load Evaluation Dataset ──────────────────────────────────────────────────

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

df_all = pd.DataFrame(raw_data)

required_cols = {"question", "answer"}
missing = required_cols - set(df_all.columns)
if missing:
    raise ValueError(f"Dataset is missing required columns: {missing}")

eval_source_df = df_all[["question", "answer"]].dropna().reset_index(drop=True)
eval_source_df = eval_source_df.iloc[:EVAL_SAMPLE_SIZE].copy()

print(f"✓ Loaded source dataset: {len(df_all)} rows")
print(f"✓ Evaluation subset: {len(eval_source_df)} rows")
display(eval_source_df.head(3))

✓ Loaded source dataset: 276139 rows
✓ Evaluation subset: 100 rows


,question,answer
0,What are the risk factors associated with deve...,Patients with two or three or more comorbiditi...
1,What are the complications associated with end...,The most common complication of EVAR is an end...
2,What are the limitations of ultrasonography in...,Ultrasonography is a cost-effective and reprod...


In [8]:
# ─── Utility ──────────────────────────────────────────────────────────────────

def l2_normalize(vec: np.ndarray) -> np.ndarray:
    vec  = np.asarray(vec, dtype=np.float32)
    norm = np.linalg.norm(vec)
    return vec if norm == 0 else vec / norm

In [9]:
# ─── Build FAISS Index if Needed ──────────────────────────────────────────────

def build_faiss_index_from_dataset(
    data_path: str,
    instructor_model_path: str,
    vectorstore_dir: str,
    limit: Optional[int] = None,
):
    os.makedirs(os.path.dirname(vectorstore_dir), exist_ok=True)

    with open(data_path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    if limit is not None:
        raw = raw[:limit]

    instruction = "Represent the cardiology answer for retrieval:"
    documents   = []

    for row in raw:
        answer = row.get("answer")
        if isinstance(answer, str) and answer.strip():
            documents.append(
                LC_Document(
                    page_content=f"{instruction}\n{answer}",
                    metadata=row,
                )
            )

    if not documents:
        raise ValueError("No valid answer documents found for indexing.")

    print(f"✓ Documents prepared for indexing: {len(documents)}")

    embeddings_model = HuggingFaceEmbeddings(
        model_name=instructor_model_path,
        model_kwargs={"device": device},
        encode_kwargs={"normalize_embeddings": False},
    )

    print("Building FAISS index...")
    vectorstore = FAISS.from_documents(documents, embeddings_model)
    vectorstore.save_local(vectorstore_dir)
    print(f"✓ FAISS index saved to: {vectorstore_dir}")


# Create or load vectorstore
if not os.path.exists(VECTORSTORE_DIR):
    print("FAISS index not found. Building a new index...")
    build_faiss_index_from_dataset(
        data_path=DATA_PATH,
        instructor_model_path=INSTRUCTOR_PATH,
        vectorstore_dir=VECTORSTORE_DIR,
        limit=None,
    )
else:
    print("✓ Existing FAISS index found")

✓ Existing FAISS index found


In [10]:
# ─── Load Vectorstore ─────────────────────────────────────────────────────────

@lru_cache(maxsize=1)
def load_vectorstore() -> FAISS:
    embeddings_model = HuggingFaceEmbeddings(
        model_name=INSTRUCTOR_PATH,
        model_kwargs={"device": device},
        encode_kwargs={"normalize_embeddings": False},
    )
    vectorstore = FAISS.load_local(
        VECTORSTORE_DIR,
        embeddings_model,
        allow_dangerous_deserialization=True,
    )
    return vectorstore


vectorstore = load_vectorstore()
print("✓ Vectorstore loaded")
try:
    print("FAISS total vectors:", vectorstore.index.ntotal)
    print("FAISS dimension:",     vectorstore.index.d)
except Exception:
    pass

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

✓ Vectorstore loaded
FAISS total vectors: 10000
FAISS dimension: 768


In [11]:
# ─── Load InBEDDER + Projection Head ──────────────────────────────────────────

class ProjectionLayer(nn.Module):
    def __init__(self, in_dim: int = 1024, out_dim: int = 768):
        super().__init__()
        self.proj = nn.Linear(in_dim, out_dim, bias=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.proj(x)


print("Loading InBEDDER model...")
inbed_model = SentenceTransformer(INBEDDER_PATH, device=device)

print("Loading projection head...")
projector = ProjectionLayer()
projector.load_state_dict(torch.load(PROJECTOR_PATH, map_location=device))
projector.to(device)
projector.eval()


def embed_and_project(query: str) -> np.ndarray:
    instruction = "Cluster this cardiology question semantically:"
    text = f"{instruction}\n{query}"

    emb = inbed_model.encode(
        text,
        convert_to_numpy=True,
        normalize_embeddings=False,
    ).astype(np.float32)

    emb = l2_normalize(emb)

    with torch.no_grad():
        x         = torch.from_numpy(emb).unsqueeze(0).to(device)
        projected = projector(x).squeeze(0)

    return projected.detach().cpu().numpy()


test_vec = embed_and_project("What is the treatment for atrial fibrillation?")
print("✓ Projected query shape:", test_vec.shape)

Loading InBEDDER model...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading projection head...
✓ Projected query shape: (768,)


In [12]:
# ─── Load HyDE Generator + Instructor Embeddings ──────────────────────────────

print("Loading HyDE generator...")
hyde_tokenizer = AutoTokenizer.from_pretrained(HYDE_PATH)
hyde_model     = AutoModelForSeq2SeqLM.from_pretrained(HYDE_PATH).to(device)
hyde_model.eval()

print("Loading Instructor embeddings for HyDE docs...")
hyde_embeddings_model = HuggingFaceEmbeddings(
    model_name=INSTRUCTOR_PATH,
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": False},
)


def generate_hypothetical_docs(
    query: str,
    num_return_sequences: int = 4,
    max_new_tokens: int = 300,
):
    instruction = "Represent the cardiology document for retrieval:"

    # IMPROVED: richer prompt for better hypothesis quality
    prompt = (
        f"Generate a detailed cardiology answer paragraph that directly addresses "
        f"the following clinical question.\n\nQuestion: {query}\n\nAnswer paragraph:"
    )

    inputs = hyde_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
    ).to(device)

    with torch.no_grad():
        outputs = hyde_model.generate(
            **inputs,
            do_sample=True,
            num_beams=4,
            num_return_sequences=num_return_sequences,
            temperature=0.7,
            max_new_tokens=max_new_tokens,
        )

    replies = [
        hyde_tokenizer.decode(out, skip_special_tokens=True)
        for out in outputs
    ]

    docs = [f"{instruction}\n{reply}" for reply in replies]

    embeddings = np.array(
        hyde_embeddings_model.embed_documents(docs),
        dtype=np.float32,
    )

    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    embeddings = embeddings / norms

    avg_embedding = l2_normalize(np.mean(embeddings, axis=0))
    return avg_embedding, docs


hyde_test_emb, hyde_test_docs = generate_hypothetical_docs("What causes myocardial infarction?")
print("✓ HyDE embedding shape:", hyde_test_emb.shape)
print("✓ Sample hypothesis preview:")
print(hyde_test_docs[0][:250])

Loading HyDE generator...


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Loading Instructor embeddings for HyDE docs...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

✓ HyDE embedding shape: (768,)
✓ Sample hypothesis preview:
Represent the cardiology document for retrieval:
Generate a detailed cardiology answer paragraph that directly addresses the following clinical question: What causes myocardial infarction? Answer paragraph: What causes myocardial infarction?


In [13]:
# ─── Fusion + Domain Gate + Retrieval ─────────────────────────────────────────

def fuse_embeddings(
    proj_query: np.ndarray,
    hyde_query: np.ndarray,
    alpha: float = 0.5,
) -> np.ndarray:
    fused = alpha * proj_query + (1 - alpha) * hyde_query
    return l2_normalize(fused)


def domain_gate_text(
    vectorstore: FAISS,
    query: str,
    max_distance: float = DOMAIN_MAX_DISTANCE_TEXT,
):
    try:
        res = vectorstore.similarity_search_with_score(query=query, k=1)
    except Exception:
        return True, None

    if not res:
        return False, None

    best_score = res[0][1]
    try:
        best_score = float(best_score)
    except Exception:
        return True, None

    return (best_score <= max_distance), best_score


def retrieve_docs_by_vector(
    vectorstore: FAISS,
    query_embedding: np.ndarray,
    k: int = 5,
):
    query_embedding = l2_normalize(query_embedding).astype(np.float32)
    return vectorstore.similarity_search_with_score_by_vector(
        embedding=query_embedding.tolist(),
        k=k,
    )


# IMPROVED: remove near-duplicate retrieved passages
def deduplicate_docs(
    docs_and_scores: List[Tuple[LC_Document, float]],
    sim_threshold: float = 0.92,
) -> List[Tuple[LC_Document, float]]:
    """Drop retrieved passages that heavily overlap with a higher-ranked one."""
    def trigrams(text):
        words = text.lower().split()
        return set(zip(words, words[1:], words[2:]))

    kept = []
    for doc, score in docs_and_scores:
        tg_new   = trigrams(doc.page_content)
        duplicate = False
        for kept_doc, _ in kept:
            tg_kept = trigrams(kept_doc.page_content)
            union   = len(tg_new | tg_kept)
            if union == 0:
                continue
            if len(tg_new & tg_kept) / union >= sim_threshold:
                duplicate = True
                break
        if not duplicate:
            kept.append((doc, score))
    return kept


print("✓ Retrieval functions ready")

✓ Retrieval functions ready


In [14]:
# ─── Load Llama-3.2-3B-Instruct (Gated Model) ─────────────────────────────────
#
# Uses 4-bit quantization (NF4) to fit on a free Colab T4 GPU.
# HF login (cell above) is required to access this gated model.

print("Loading Llama-3.2-3B-Instruct tokenizer...")
llama_tokenizer = AutoTokenizer.from_pretrained(
    FINAL_LLM_MODEL,
    token=HF_TOKEN,
)
llama_tokenizer.pad_token = llama_tokenizer.eos_token

print("Loading Llama-3.2-3B-Instruct model (4-bit quantized)...")
try:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    llama_model = AutoModelForCausalLM.from_pretrained(
        FINAL_LLM_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        token=HF_TOKEN,
    ).eval()
    print("✓ Llama loaded with 4-bit quantization")
except Exception as e:
    print(f"⚠️  4-bit quant failed ({e}), falling back to fp16...")
    llama_model = AutoModelForCausalLM.from_pretrained(
        FINAL_LLM_MODEL,
        torch_dtype=torch.float16,
        device_map="auto",
        token=HF_TOKEN,
    ).eval()
    print("✓ Llama loaded in fp16")

print("✓ Llama-3.2-3B-Instruct ready")

Loading Llama-3.2-3B-Instruct tokenizer...


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

✓ Llama loaded with 4-bit quantization
✓ Llama-3.2-3B-Instruct ready


In [15]:
# ─── Final Answer Generator ───────────────────────────────────────────────────

def build_final_answer_prompt(query: str, retrieved_docs: List[LC_Document]) -> List[dict]:
    """IMPROVED: Structured chat-style prompt with numbered sources."""

    # Strip the instructor prefix added during indexing
    def clean_content(text: str) -> str:
        lines = text.split("\n", 1)
        return lines[1].strip() if len(lines) > 1 else text.strip()

    context_parts = []
    for idx, doc in enumerate(retrieved_docs, start=1):
        content = clean_content(doc.page_content)
        context_parts.append(f"[Source {idx}]\n{content}")

    context = "\n\n".join(context_parts)

    system_msg = (
        "You are an expert cardiology assistant. "
        "Answer the clinical question using ONLY the provided sources. "
        "Be medically precise, complete, and concise. "
        "If the sources do not contain enough information, say: "
        '"I cannot answer based on the provided context."'
    )

    user_msg = (
        f"Sources:\n{context}\n\n"
        f"Clinical Question: {query}\n\n"
        "Provide a thorough, accurate answer based strictly on the sources above."
    )

    return [
        {"role": "system", "content": system_msg},
        {"role": "user",   "content": user_msg},
    ]


def generate_final_answer_llama(query: str, retrieved_docs: List[LC_Document]) -> str:
    if not retrieved_docs:
        return "No relevant documents found."

    messages = build_final_answer_prompt(query, retrieved_docs)

    # Apply Llama-3's native chat template
    prompt_text = llama_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = llama_tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=3000,
    )
    inputs = {k: v.to(llama_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = llama_model.generate(
            **inputs,
            max_new_tokens=FINAL_LLM_MAX_NEW_TOKENS,
            do_sample=False,
            temperature=1.0,
            repetition_penalty=1.1,
            eos_token_id=llama_tokenizer.eos_token_id,
            pad_token_id=llama_tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens (skip the prompt)
    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    answer     = llama_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return answer if answer else "I cannot answer based on the provided context."


def generate_out_of_domain_answer(query: str) -> str:
    return (
        "This question appears to be outside the cardiology knowledge domain "
        "covered by the indexed dataset, so I cannot answer it reliably."
    )


def generate_final_answer(query: str, retrieved_docs: List[LC_Document]) -> str:
    return generate_final_answer_llama(query, retrieved_docs)


print("✓ Final answer generator set to Llama-3.2-3B-Instruct")

✓ Final answer generator set to Llama-3.2-3B-Instruct


In [16]:
# ─── Full Pipeline Function ────────────────────────────────────────────────────

def run_medirag_pipeline(
    question: str,
    k: int = TOP_K,
    alpha: float = ALPHA,
    domain_max_distance_text: float = DOMAIN_MAX_DISTANCE_TEXT,
):
    vectorstore = load_vectorstore()

    # Domain gate
    in_domain, best_score = domain_gate_text(
        vectorstore=vectorstore,
        query=question,
        max_distance=domain_max_distance_text,
    )

    if not in_domain:
        return {
            "answer": generate_out_of_domain_answer(question),
            "retrieved_answers": [],
            "debug": {
                "domain_in_domain": False,
                "domain_best_score_text": best_score,
            },
        }

    # Dual embeddings
    proj_emb = l2_normalize(np.array(embed_and_project(question)))
    hyde_emb, hyde_docs = generate_hypothetical_docs(question)
    hyde_emb = l2_normalize(np.array(hyde_emb))

    # Fusion
    final_emb = l2_normalize(np.array(
        fuse_embeddings(proj_query=proj_emb, hyde_query=hyde_emb, alpha=alpha)
    ))

    # Retrieve
    docs_and_scores = retrieve_docs_by_vector(
        vectorstore=vectorstore,
        query_embedding=final_emb,
        k=k,
    )

    # IMPROVED: deduplicate
    docs_and_scores = deduplicate_docs(docs_and_scores)

    retrieved_docs = [doc for doc, _ in docs_and_scores]
    answer         = generate_final_answer(question, retrieved_docs)

    retrieved_answers = []
    for doc, score in docs_and_scores:
        item = dict(doc.metadata) if isinstance(doc.metadata, dict) else {}
        item["text"]  = doc.page_content
        item["score"] = float(score)
        retrieved_answers.append(item)

    return {
        "answer": answer,
        "retrieved_answers": retrieved_answers,
        "hyde_docs": hyde_docs,
        "debug": {
            "domain_in_domain": True,
            "domain_best_score_text": best_score,
        },
    }


# Quick smoke test
sample_question = eval_source_df.iloc[0]["question"]
sample_result   = run_medirag_pipeline(sample_question)
print("Question:", sample_question)
print("\nAnswer preview:", sample_result["answer"][:500])
print("\nRetrieved docs:", len(sample_result["retrieved_answers"]))

The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Question: What are the risk factors associated with developing postoperative heart failure in patients with comorbidities?

Answer preview: Based on the provided sources, the risk factors associated with developing postoperative heart failure in patients with comorbidities can be inferred as follows:

According to Source 1, having two or three or more comorbidities on admission significantly increases the risk of developing postoperative heart failure compared to those with no comorbidities.

Additionally, Source 2 lists hypertension, ischemic heart disease, and diabetes as risk factors for heart failure. Since these conditions ofte

Retrieved docs: 5


In [23]:
# Force fresh run — delete existing checkpoint
import os
if os.path.exists(PIPELINE_OUTPUT_PATH):
    os.remove(PIPELINE_OUTPUT_PATH)
    print("🗑️  Deleted existing checkpoint, starting fresh")

🗑️  Deleted existing checkpoint, starting fresh


In [24]:
# ─── Evaluation Data Collection Loop with Checkpointing ──────────────────────

results = []

if os.path.exists(PIPELINE_OUTPUT_PATH):
    existing            = pd.read_csv(PIPELINE_OUTPUT_PATH)
    completed_questions = set(existing["question"].tolist())
    results             = existing.to_dict("records")
    print(f"✓ Resuming from checkpoint: {len(results)} already done")
else:
    completed_questions = set()
    print("✓ Starting fresh run")

remaining_df = eval_source_df[
    ~eval_source_df["question"].isin(completed_questions)
].reset_index(drop=True)
print(f"Remaining questions: {len(remaining_df)}")

for i, row in tqdm(remaining_df.iterrows(), total=len(remaining_df), desc="Running pipeline"):
    question  = row["question"]
    reference = row["answer"]

    try:
        result = run_medirag_pipeline(question)

        contexts = [
            c.get("answer", c.get("text", ""))
            for c in result.get("retrieved_answers", [])
        ]

        results.append({
            "question":  question,
            "answer":    result["answer"],
            "contexts":  json.dumps(contexts, ensure_ascii=False),
            "reference": reference,
        })

    except Exception as e:
        print(f"\n✗ Failed on row {i}: {question[:80]}...")
        print("Error:", e)
        results.append({
            "question":  question,
            "answer":    "PIPELINE_ERROR",
            "contexts":  "[]",
            "reference": reference,
        })

    if (len(results) % 10) == 0:
        pd.DataFrame(results).to_csv(PIPELINE_OUTPUT_PATH, index=False)
        print(f"💾 Checkpoint saved at {len(results)} rows")

eval_df = pd.DataFrame(results)
eval_df.to_csv(PIPELINE_OUTPUT_PATH, index=False)

print(f"\n✓ Done! {len(eval_df)} rows saved to {PIPELINE_OUTPUT_PATH}")
print(f"Pipeline errors: {(eval_df['answer'] == 'PIPELINE_ERROR').sum()}")
display(eval_df.head(2))

✓ Starting fresh run
Remaining questions: 100


Running pipeline:   0%|          | 0/100 [00:00<?, ?it/s]

💾 Checkpoint saved at 10 rows
💾 Checkpoint saved at 20 rows
💾 Checkpoint saved at 30 rows
💾 Checkpoint saved at 40 rows
💾 Checkpoint saved at 50 rows
💾 Checkpoint saved at 60 rows
💾 Checkpoint saved at 70 rows
💾 Checkpoint saved at 80 rows
💾 Checkpoint saved at 90 rows
💾 Checkpoint saved at 100 rows

✓ Done! 100 rows saved to /content/drive/MyDrive/Research_My_Component/cardiology/eval_results_pipeline_first100.csv
Pipeline errors: 0


,question,answer,contexts,reference
0,What are the risk factors associated with deve...,"Based on the provided sources, the risk factor...","[""Patients with two or three or more comorbidi...",Patients with two or three or more comorbiditi...
1,What are the complications associated with end...,"Based on the provided sources, the complicatio...","[""EVAR (endovascular aneurysm repair) has evol...",The most common complication of EVAR is an end...


In [25]:
# ─── Quick Sanity Checks ──────────────────────────────────────────────────────

def safe_context_preview(ctx_str):
    try:
        parsed = json.loads(ctx_str)
        if parsed:
            return parsed[0][:200]
        return "EMPTY"
    except Exception:
        return "INVALID_CONTEXT_JSON"

print(f"Total rows: {len(eval_df)}")
print(f"Pipeline errors: {(eval_df['answer'] == 'PIPELINE_ERROR').sum()}")
print(f"Empty answers: {(eval_df['answer'] == '').sum()}")
print(f"Empty contexts: {(eval_df['contexts'] == '[]').sum()}")

print("\n--- Sample Row ---")
sample = eval_df.iloc[0]
print("Question :", sample["question"][:120])
print("Answer   :", sample["answer"][:200])
print("Reference:", sample["reference"][:200])
print("Contexts :", safe_context_preview(sample["contexts"]))

Total rows: 100
Pipeline errors: 0
Empty answers: 0
Empty contexts: 0

--- Sample Row ---
Question : What are the risk factors associated with developing postoperative heart failure in patients with comorbidities?
Answer   : Based on the provided sources, the risk factors associated with developing postoperative heart failure in patients with comorbidities can be inferred as follows:

According to Source 1, having two or 
Reference: Patients with two or three or more comorbidities on admission have an increased risk of developing postoperative heart failure compared to those with no comorbidity. The risk of postoperative heart fa
Contexts : Patients with two or three or more comorbidities on admission have an increased risk of developing postoperative heart failure compared to those with no comorbidity. The risk of postoperative heart fa


In [26]:
# ─── Prepare RAGAS Dataset ────────────────────────────────────────────────────

def parse_contexts(ctx_str):
    try:
        parsed = json.loads(ctx_str)
        return parsed if parsed else ["no context retrieved"]
    except Exception:
        return ["no context retrieved"]


ragas_data = {
    "question":  eval_df["question"].tolist(),
    "answer":    eval_df["answer"].tolist(),
    "contexts":  [parse_contexts(c) for c in eval_df["contexts"]],
    "reference": eval_df["reference"].tolist(),
}

ragas_dataset = Dataset.from_dict(ragas_data)
print(f"✓ RAGAS dataset ready: {len(ragas_dataset)} rows")
print("Sample contexts type:", type(ragas_dataset[0]["contexts"]))

✓ RAGAS dataset ready: 100 rows
Sample contexts type: <class 'list'>


In [27]:
# ─── RAGAS Evaluation ─────────────────────────────────────────────────────────

judge_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
metrics = [
    AnswerCorrectness(llm=judge_llm),
    Faithfulness(llm=judge_llm),
    AnswerRelevancy(llm=judge_llm),
]
print("✓ GPT-4o-mini configured for RAGAS judging")

print("Running RAGAS evaluation...")
ragas_results = evaluate(
    dataset=ragas_dataset,
    metrics=metrics,
)

ragas_df = ragas_results.to_pandas()
print("✓ RAGAS scoring complete")
display(ragas_df[["answer_correctness", "faithfulness", "answer_relevancy"]].describe())

/tmp/ipykernel_715/1937315667.py:3: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  judge_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))


✓ GPT-4o-mini configured for RAGAS judging
Running RAGAS evaluation...


Evaluating:   0%|          | 0/300 [00:00<?, ?it/s]

✓ RAGAS scoring complete


,answer_correctness,faithfulness,answer_relevancy
count,100.000000,100.000000,100.000000
mean,0.648763,0.829663,0.584104
std,0.207236,0.184154,0.479725
min,0.216479,0.000000,0.000000
25%,0.493625,0.743056,0.000000
50%,0.633897,0.870833,0.952597
75%,0.814165,1.000000,0.986282
max,0.997644,1.000000,1.000000


In [28]:
# ─── LLM-as-Judge ─────────────────────────────────────────────────────────────

def llm_judge(question, pipeline_answer, reference_answer):
    prompt = f"""You are a medical AI evaluator. Score the pipeline answer against the reference answer.

Question: {question}

Reference Answer: {reference_answer}

Pipeline Answer: {pipeline_answer}

Score on these two dimensions (1-5 each):

1. COMPLETENESS: Does the pipeline answer cover all key points in the reference answer?
   1 = Covers almost nothing
   3 = Covers some key points
   5 = Fully covers all key points

2. HALLUCINATION: Does the pipeline answer introduce facts NOT in the reference answer?
   1 = Major unsupported claims
   3 = Minor unsupported additions
   5 = No hallucinations, fully grounded

Respond ONLY in this exact JSON format:
{{\"completeness\": <1-5>, \"hallucination\": <1-5>, \"reason\": \"<one sentence>\"}}"""

    try:
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=150,
        )
        raw = response.choices[0].message.content.strip()
        return json.loads(raw)
    except Exception as e:
        print(f"  ✗ Judge failed: {e}")
        return {"completeness": None, "hallucination": None, "reason": "error"}


print("Running LLM-as-Judge scoring...")
judge_scores = []

for i, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Judging"):
    score = llm_judge(row["question"], row["answer"], row["reference"])
    judge_scores.append(score)

judge_df = pd.DataFrame(judge_scores)
print("✓ LLM-as-Judge scoring complete")
display(judge_df[["completeness", "hallucination"]].describe())

Running LLM-as-Judge scoring...


Judging:   0%|          | 0/100 [00:00<?, ?it/s]

✓ LLM-as-Judge scoring complete


,completeness,hallucination
count,100.000000,100.000000
mean,4.510000,4.920000
std,0.958692,0.367355
min,1.000000,3.000000
25%,4.750000,5.000000
50%,5.000000,5.000000
75%,5.000000,5.000000
max,5.000000,5.000000


In [29]:
# ─── Merge Final Scores & Save ────────────────────────────────────────────────

final_df = eval_df.copy()
final_df["ragas_answer_correctness"] = ragas_df["answer_correctness"].values
final_df["ragas_faithfulness"]       = ragas_df["faithfulness"].values
final_df["ragas_answer_relevancy"]   = ragas_df["answer_relevancy"].values
final_df["judge_completeness"]       = judge_df["completeness"].values
final_df["judge_hallucination"]      = judge_df["hallucination"].values
final_df["judge_reason"]             = judge_df["reason"].values

final_df.to_csv(FINAL_OUTPUT_PATH, index=False)
print(f"✓ Final scored results saved to {FINAL_OUTPUT_PATH}")

print("\n===== PIPELINE EVALUATION SUMMARY =====")
print(f"RAGAS Answer Correctness : {final_df['ragas_answer_correctness'].mean():.3f} / 1.0")
print(f"RAGAS Faithfulness       : {final_df['ragas_faithfulness'].mean():.3f} / 1.0")
print(f"RAGAS Answer Relevancy   : {final_df['ragas_answer_relevancy'].mean():.3f} / 1.0")
print(f"Judge Completeness       : {final_df['judge_completeness'].mean():.2f} / 5.0")
print(f"Judge Hallucination      : {final_df['judge_hallucination'].mean():.2f} / 5.0")

display(final_df.head(3))

✓ Final scored results saved to /content/drive/MyDrive/Research_My_Component/cardiology/eval_final_scored_first100.csv

===== PIPELINE EVALUATION SUMMARY =====
RAGAS Answer Correctness : 0.649 / 1.0
RAGAS Faithfulness       : 0.830 / 1.0
RAGAS Answer Relevancy   : 0.584 / 1.0
Judge Completeness       : 4.51 / 5.0
Judge Hallucination      : 4.92 / 5.0


,question,answer,contexts,reference,ragas_answer_correctness,ragas_faithfulness,ragas_answer_relevancy,judge_completeness,judge_hallucination,judge_reason
0,What are the risk factors associated with deve...,"Based on the provided sources, the risk factor...","[""Patients with two or three or more comorbidi...",Patients with two or three or more comorbiditi...,0.525418,0.818182,1.000000,4,5,The pipeline answer covers the key points abou...
1,What are the complications associated with end...,"Based on the provided sources, the complicatio...","[""EVAR (endovascular aneurysm repair) has evol...",The most common complication of EVAR is an end...,0.695027,0.687500,0.000000,4,4,The pipeline answer covers most key points fro...
2,What are the limitations of ultrasonography in...,"Based on Source 1, ultrasonography has a limit...","[""Ultrasonography is a cost-effective and repr...",Ultrasonography is a cost-effective and reprod...,0.689468,0.800000,0.973533,5,5,The pipeline answer fully covers the key point...
